# Lab 3: Agentes no Azure AI Foundry (15 min)

## Objetivos
- Criar um agente com o SDK `azure-ai-projects`
- Usar function calling (tools)
- Gerir conversas via threads
- Entender o ciclo: Agent → Thread → Run

## Conceitos

### O que é um Agente?
Um **Agente** no Foundry é um assistente inteligente que pode:
- Usar **ferramentas** (code interpreter, file search, functions)
- Manter **contexto** ao longo de uma conversa (thread)
- **Raciocinar** sobre que ferramenta usar e quando

### Ciclo de Vida
```
1. Criar Agente (com instruções e tools)
2. Criar Thread (conversa)
3. Adicionar Mensagem ao Thread
4. Criar Run (execução)
5. Processar Resposta (pode incluir tool calls)
```

### SDK
```python
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential

project = AIProjectClient(
    endpoint=FOUNDRY_ENDPOINT,
    credential=AzureKeyCredential(FOUNDRY_KEY),
)
```

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FunctionTool, ToolSet
from azure.core.credentials import AzureKeyCredential

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

project = AIProjectClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

print(f"✅ Projeto conectado: {endpoint}")

## 🖥️ Exercício 3.1: Agente Simples

Vamos criar um agente básico sem ferramentas - apenas com instruções.

In [ ]:
# Exercício 3.1: Criar e conversar com um agente simples
agent = project.agents.create_agent(
    model=model,
    name="assistente-azure",
    instructions="És um especialista em Azure. Responde de forma concisa em português de Portugal. Usa exemplos práticos.",
)
print(f"✅ Agente criado: {agent.id}")

# Criar thread e enviar mensagem
thread = project.agents.threads.create()
print(f"   Thread: {thread.id}")

project.agents.messages.create(
    thread_id=thread.id,
    role="user",
    content="Explica a diferença entre Azure Functions e Container Apps em 3 linhas.",
)

# Executar
run = project.agents.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)
print(f"   Run status: {run.status}")

# Obter resposta
messages = project.agents.messages.list(thread_id=thread.id)
for msg in reversed(messages.data):
    role = "👤" if msg.role == "user" else "🤖"
    for item in msg.content:
        if hasattr(item, "text"):
            print(f"{role} {item.text.value}")

## 🖥️ Exercício 3.2: Agente com Function Calling

O verdadeiro poder dos agentes é usar **ferramentas**. Vamos criar um agente que pode consultar informações.

In [ ]:
# Exercício 3.2: Agente com function tools
import json

# Definir funções que o agente pode chamar
def obter_preco_servico(servico: str) -> str:
    """Retorna o preço estimado de um serviço Azure."""
    precos = {
        "app service basic": "€50/mês",
        "container apps": "€0.000012/vCPU-s",
        "functions consumption": "€0.000016/GB-s",
        "aks": "Grátis (control plane) + custo dos nodes",
        "cosmos db": "A partir de €25/mês (serverless)",
    }
    return json.dumps({"servico": servico, "preco": precos.get(servico.lower(), "Preço não disponível. Consulta azure.com/pricing")})

def obter_regiao_recomendada(caso_uso: str) -> str:
    """Recomenda a melhor região Azure para um caso de uso."""
    regioes = {
        "europa": "West Europe (Holanda) ou North Europe (Irlanda)",
        "portugal": "West Europe (mais próximo de Portugal)",
        "global": "Usa Traffic Manager com múltiplas regiões",
        "ai": "East US ou Sweden Central (melhor disponibilidade de modelos)",
    }
    return json.dumps({"caso_uso": caso_uso, "recomendacao": regioes.get(caso_uso.lower(), regioes["europa"])})

# Registar as funções como tools
functions = FunctionTool(functions=[
    {
        "name": "obter_preco_servico",
        "description": "Obtém o preço estimado de um serviço Azure",
        "parameters": {
            "type": "object",
            "properties": {
                "servico": {"type": "string", "description": "Nome do serviço Azure (ex: app service basic, container apps)"}
            },
            "required": ["servico"]
        }
    },
    {
        "name": "obter_regiao_recomendada",
        "description": "Recomenda a melhor região Azure para um caso de uso",
        "parameters": {
            "type": "object",
            "properties": {
                "caso_uso": {"type": "string", "description": "Caso de uso (europa, portugal, global, ai)"}
            },
            "required": ["caso_uso"]
        }
    },
])

toolset = ToolSet()
toolset.add(functions)

print("✅ Funções definidas: obter_preco_servico, obter_regiao_recomendada")

In [ ]:
# Criar agente com tools e testar
agent_tools = project.agents.create_agent(
    model=model,
    name="consultor-azure",
    instructions="És um consultor Azure. Usa as ferramentas disponíveis para dar informações precisas sobre preços e regiões. Responde em português.",
    toolset=toolset,
)

thread2 = project.agents.threads.create()
project.agents.messages.create(
    thread_id=thread2.id,
    role="user",
    content="Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?",
)

# Executar - o SDK processa automaticamente os tool calls
run2 = project.agents.runs.create_and_process(
    thread_id=thread2.id,
    agent_id=agent_tools.id,
    tool_resources={
        "obter_preco_servico": obter_preco_servico,
        "obter_regiao_recomendada": obter_regiao_recomendada,
    },
)

# Mostrar resultado
messages2 = project.agents.messages.list(thread_id=thread2.id)
for msg in reversed(messages2.data):
    role = "👤" if msg.role == "user" else "🤖"
    for item in msg.content:
        if hasattr(item, "text"):
            print(f"{role} {item.text.value}\n")

## 🖥️ Exercício 3.3: Code Interpreter

O **Code Interpreter** permite ao agente escrever e executar código Python.

In [ ]:
# Exercício 3.3: Agente com Code Interpreter
from azure.ai.projects.models import CodeInterpreterTool

agent_code = project.agents.create_agent(
    model=model,
    name="analista-dados",
    instructions="És um analista de dados. Usa o code interpreter para fazer cálculos e criar visualizações. Responde em português.",
    tools=CodeInterpreterTool().definitions,
)

thread3 = project.agents.threads.create()
project.agents.messages.create(
    thread_id=thread3.id,
    role="user",
    content="Calcula a sequência de Fibonacci até ao 10º número e mostra o resultado numa tabela formatada.",
)

run3 = project.agents.runs.create_and_process(
    thread_id=thread3.id,
    agent_id=agent_code.id,
)

messages3 = project.agents.messages.list(thread_id=thread3.id)
for msg in reversed(messages3.data):
    if msg.role == "assistant":
        for item in msg.content:
            if hasattr(item, "text"):
                print(f"🤖 {item.text.value}")

In [ ]:
# Limpeza - apagar agentes criados
for a in [agent, agent_tools, agent_code]:
    project.agents.delete_agent(a.id)
    print(f"🗑️ Agente {a.name} eliminado")

print("\n✅ Limpeza concluída!")

## ✅ Resumo

Neste lab aprendeste:
- O ciclo Agent → Thread → Message → Run
- Criar agentes simples com instruções
- Usar **function calling** para dar acesso a dados externos
- Usar o **Code Interpreter** para computação

**Próximo:** [Lab 4 - Knowledge e RAG](lab04-knowledge-rag.ipynb)